В этом ноутбуке детально показан процесс обучения NL2L модуля.

NL2L модуль - это seq2seq модель, которая принимает на вход текстовое описание математического выражения и возвращает код на LaTeX, соответствующий этому математическому выражению

In [ ]:
!pip install -q transformers datasets accelerate evaluate rouge_score

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Загружаем датасеты для обучения и валидации

In [ ]:
import pandas as pd
from datasets import Dataset, DatasetDict
from google.colab import files
import os

for filename in ['s2l_train.csv', 's2l_test.csv']:
    if not os.path.exists(filename):
        print(f"Пожалуйста, выберите и загрузите файл {filename}:")
        files.upload()

# Загружаем CSV в pandas и переводим в формат Hugging Face Dataset
train_df = pd.read_csv('s2l_train.csv')
test_df = pd.read_csv('s2l_test.csv')

raw_datasets = DatasetDict({
    'train': Dataset.from_pandas(train_df),
    'test': Dataset.from_pandas(test_df)
})

print(f"Размер обучающей выборки: {len(raw_datasets['train'])}")
print(f"Размер тестовой выборки: {len(raw_datasets['test'])}")

Размер обучающей выборки: 20477
Размер тестовой выборки: 2276


Инициализируем токенизатор для ruT5-base

In [ ]:
from transformers import AutoTokenizer

model_checkpoint = "ai-forever/ruT5-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

Некоторым редковстречающимся токенам токенизатор присваивает id=1 (special token). В частности, такой id присваивается многим математическим спецсимволам, что нехорошо, потому что все эти символы сливаются в один и тот же id. Чтобы избежать этого, явно скажем токенизатору, что эти символы необходимо помечать разными id

In [ ]:
special_latex_tokens = [
    "{", "}", "}{",                                     # Фигурные скобки и их стык
    "<", ">",                                           # Знаки сравнения
    "|",                                                # Модуль или логическое ИЛИ
    "&",                                                # Амперсанд
    "^",                                                # степень
    "~",                                                # Неразрывный пробел
    "#",                                                # Знак решетки
    "α", "Α", "β", "Β", "γ", "Γ", "δ", "Δ", "ε", "Ε",
    "ζ", "Ζ", "η", "Η", "θ", "Θ", "ι", "Ι", "κ", "Κ",
    "λ", "Λ", "μ", "Μ", "ν", "Ν", "ξ", "Ξ", "π", "Π",
    "ρ", "Ρ", "σ", "Σ", "τ", "Т", "υ", "Υ", "φ", "Φ",
    "χ", "Χ", "ψ", "Ψ", "ω", "Ω"                        # Греческие буквы
]

tokenizer.add_tokens(special_latex_tokens)

56

Превращаем текстовые строки в числовые векторы (id токенов), которые понимает модель. Ограничиваем максимальную длину в 128 токенов (этого с запасом хватит для большинства формул).

In [ ]:
max_input_length = max_target_length = 128

def preprocess_function(examples):
    model_inputs = tokenizer(examples["pronunciation"], max_length=max_input_length, truncation=True)

    labels = tokenizer(text_target=examples["sentence_normalized"], max_length=max_target_length, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = raw_datasets.map(preprocess_function, batched=True, remove_columns=raw_datasets["train"].column_names)
print("Токенизация успешно завершена!")


Map:   0%|          | 0/20477 [00:00<?, ? examples/s]

Map:   0%|          | 0/2276 [00:00<?, ? examples/s]

Токенизация успешно завершена!


Вводим метрики

In [ ]:
import evaluate
import numpy as np

metric = evaluate.load("rouge")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        # Корректно извлекаем массив предсказаний, если он пришел в виде кортежа
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [label.strip() for label in decoded_labels]

    result = metric.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    result = {k: round(v * 100, 4) for k, v in result.items()}

    return result



Ну, поехали

In [ ]:
from transformers import (
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

model_checkpoint = "ai-forever/ruT5-base"

model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint, use_safetensors=True)

model.gradient_checkpointing_enable()
model.resize_token_embeddings(len(tokenizer))

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="/content/drive/MyDrive/nl2l-base",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,

    learning_rate=3e-5,
    weight_decay=0.05,

    per_device_train_batch_size=8,      # увеличен до 8 (без OOM на T4)
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,      # эффективный батч = 8 * 2 = 16

    lr_scheduler_type="cosine",
    warmup_ratio=0.1,

    num_train_epochs=4,
    predict_with_generate=True,
    fp16=True,
    logging_steps=100,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    max_length=128,
    compute_metrics=compute_metrics,
)

trainer.train()


model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

In [ ]:
# Сохраняем модель локально в папку
model.save_pretrained("./nl2l-base/model")
tokenizer.save_pretrained("./nl2l-base/model")

# Архивируем папку, чтобы скачать её одним файлом
!tar -czf nl2l-base/model.tar.gz ./nl2l-base
files.download('nl2l-base/model.tar.gz')


In [ ]:
import os
import pandas as pd
import torch
from google.colab import files
from tqdm import tqdm
from IPython.display import display, HTML

if not os.path.exists('essence.csv'):
    print("Пожалуйста, выберите и загрузите файл essential.csv:")
    files.upload()

try:
    # Пробуем стандартную запятую, но разрешаем пропускать битые строки, если они есть
    essential_df = pd.read_csv('essence.csv', sep=None, engine='python', on_bad_lines='skip')
except Exception as e:
    print(f"Не удалось автоматически прочитать файл: {e}")
    print("Пробуем жестко задать точку с запятой ';'")
    essential_df = pd.read_csv('essence.csv', sep=';', on_bad_lines='skip')

if 'pronunciation' not in essential_df.columns or 'sentence_normalized' not in essential_df.columns:
    print("⚠️ Ошибка: Столбцы не найдены. Доступные столбцы в файле:", list(essential_df.columns))
    raise ValueError("Убедитесь, что имена столбцов в CSV совпадают с 'pronunciation' и 'sentence_normalized'!")

print(f"Успешно загружено строк для проверки: {len(essential_df)}")

model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

results = []

print("Запуск инференса модели по датасету essential.csv...")

for idx, row in tqdm(essential_df.iterrows(), total=len(essential_df)):
    text_input = str(row['pronunciation']).strip()
    true_latex = str(row['sentence_normalized']).strip()

    inputs = tokenizer(text_input, return_tensors="pt", max_length=128, truncation=True).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=128,
            num_beams=4,
            early_stopping=True
        )

    predicted_latex = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

    results.append({
        "Русское описание": text_input,
        "Истинный код": true_latex,
        "Истинный рендер": f"$${true_latex}$$",
        "Предсказание код": predicted_latex,
        "Предсказание рендер": f"$${predicted_latex}$$"
    })

from IPython.display import display, Math, Markdown

for i, res in enumerate(results):
    display(Markdown(f"### Пример {i+1}"))
    display(Markdown(f"**Текст:** *«{res['Русское описание']}»*"))

    display(Markdown(f"**Ожидаемый код:** `{res['Истинный код']}` | **Модель выдала:** `{res['Предсказание код']}`"))

    display(Markdown("**Визуальный рендеринг (Идеал vs Модель):**"))
    try:
        # Рисуем то, что должно быть, и то, что получилось, рядом через знак равенства или стрелку
        display(Math(f"{res['Истинный код']} \\quad \\Longrightarrow \\quad {res['Предсказание код']}"))
    except Exception:
        display(Markdown("*(Ошибка рендеринга синтаксиса модели)*"))

    display(Markdown("---"))

# Сохраняем результаты в CSV
results_df = pd.DataFrame(results)
results_df.to_csv("essential_validation_results.csv", index=False)

